In [1]:
import pandas as pd
import sqlite3
import os

DB_PATH = os.path.join(os.getcwd(), "data", "raw", "logistics.db")
conn = sqlite3.connect(DB_PATH)

# 1. On-time delivery rate по перевозчикам
q1 = """
SELECT 
    carrier,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delay_days <= 0 THEN 1 ELSE 0 END) AS on_time,
    ROUND(SUM(CASE WHEN delay_days <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS on_time_pct,
    ROUND(AVG(delay_days), 1) AS avg_delay_days
FROM deliveries
WHERE delivered = 1
GROUP BY carrier
ORDER BY on_time_pct DESC;
"""
display(pd.read_sql(q1, conn))

# 2. Факторы задержки: дистанция + день недели
q2 = """
SELECT 
    CASE 
        WHEN distance_km < 100 THEN '0-100 km'
        WHEN distance_km < 300 THEN '100-300 km'
        ELSE '300+ km'
    END AS distance_group,
    CASE CAST(strftime('%w', order_date) AS INTEGER)
        WHEN 0 THEN 'Sunday' WHEN 6 THEN 'Saturday' ELSE 'Weekday'
    END AS day_type,
    ROUND(AVG(delay_days), 1) AS avg_delay,
    COUNT(*) AS orders
FROM deliveries
WHERE delivered = 1 AND delay_days IS NOT NULL
GROUP BY 1, 2
ORDER BY avg_delay DESC;
"""
display(pd.read_sql(q2, conn))

conn.close()

,carrier,total_orders,on_time,on_time_pct,avg_delay_days
0,Russian Post,2310,1880,81.4,0.3
1,PickPoint,2347,1904,81.1,0.3
2,Boxberry,2288,1813,79.2,0.3
3,CDEK,2284,1805,79.0,0.4


,distance_group,day_type,avg_delay,orders
0,0-100 km,Saturday,0.4,384
1,0-100 km,Weekday,0.4,1835
2,100-300 km,Sunday,0.4,428
3,0-100 km,Sunday,0.3,393
4,100-300 km,Saturday,0.3,456
5,100-300 km,Weekday,0.3,2341
6,300+ km,Saturday,0.3,469
7,300+ km,Sunday,0.3,477
8,300+ km,Weekday,0.3,2446
